# 03 · De calificaciones a rankings

**Objetivo:** en la práctica no se predice “cuántas estrellas”, se decide **qué mostrar**. Este notebook
compara modelos por métricas de ranking (NDCG@10, cobertura, sesgo de popularidad), muestra por qué
el modelo con mejor RMSE puede rankear mal, e introduce la factorización para datos **implícitos**.

In [1]:
import numpy as np
from applied_recsys.baselines import BiasModel, popularity_scores
from applied_recsys.data import random_split, to_csr
from applied_recsys.synthetic import synthetic_ratings

from common import set_seed, setup_style

setup_style()
rng = set_seed(42)
from applied_recsys.factorization import FunkSVD, ImplicitALS, PureSVD
from applied_recsys.metrics import (
    catalog_coverage,
    mean_popularity_percentile,
    ranking_metrics,
    top_n,
)
from applied_recsys.neighbors import ItemKNN

df, truth = synthetic_ratings(n_users=1500, n_items=800, density=0.04, mnar=1.0, seed=2)
shape = (1500, 800)
tr, te = random_split(df, 0.25, seed=0)
Rtr = to_csr(tr, shape)
pop = popularity_scores(Rtr)
relevant = {int(u): set(g.i) for u, g in te[te.rating >= 4].groupby("u")}
users = np.array(sorted(relevant))
print(f"{len(users)} usuarios con al menos un ítem relevante (≥ 4) en test")

1445 usuarios con al menos un ítem relevante (≥ 4) en test


## 1. Comparación de modelos de ranking

In [2]:
bias = BiasModel().fit(Rtr)
funk = FunkSVD(k=10, epochs=30).fit(Rtr)
models = {
    "popularidad": lambda us: np.tile(pop, (len(us), 1)),
    "FunkSVD (rating predicho)": lambda us: (
        funk.mu + funk.bu[us, None] + funk.bi[None, :] + funk.P[us] @ funk.Q.T
    ),
    "item-kNN": ItemKNN(k=50).fit(Rtr, bias).scores,
    "PureSVD (k = 20)": PureSVD(20).fit(Rtr).scores,
    "ALS implícito": ImplicitALS(k=32, alpha=1.0, n_iter=10).fit(Rtr).scores,
}
rows = {}
for name, fn in models.items():
    recs = top_n(fn(users), Rtr[users], 10)
    m = ranking_metrics(recs, [relevant[u] for u in users], 10)
    rows[name] = (
        m["ndcg"].mean(),
        m["recall"].mean(),
        catalog_coverage(recs, shape[1]),
        mean_popularity_percentile(recs, pop),
    )
print(f"{'modelo':<28}{'NDCG@10':>9}{'recall':>8}{'cobert.':>9}{'pop. pct':>10}")
for name, (n, r, c, p) in rows.items():
    print(f"{name:<28}{n:>9.3f}{r:>8.3f}{c:>9.3f}{p:>10.1f}")

modelo                        NDCG@10  recall  cobert.  pop. pct
popularidad                     0.336   0.345    0.037      98.7
FunkSVD (rating predicho)       0.028   0.047    0.058      76.4
item-kNN                        0.141   0.150    0.705      77.8
PureSVD (k = 20)                0.113   0.141    0.122      96.1
ALS implícito                   0.235   0.242    0.274      94.8


En la evaluación habitual (ítems con ≥ 4 estrellas en el test observado), **la popularidad gana**
(0.336) y FunkSVD, el modelo de calificaciones, queda último (0.028). La cobertura cuenta otra historia:
popularidad recomienda el 4 % del catálogo; item-kNN, el 70 %.

## 2. La trampa de la evaluación offline

Con datos reales solo vemos lo que la gente calificó. Con datos sintéticos conocemos también los
gustos **reales**: comparemos la misma lista contra los 20 ítems no vistos que más le gustarían a cada usuario.

In [3]:
seen = Rtr.toarray() > 0
true_pref = np.where(seen, -np.inf, truth["true"])
relevant_true = {int(u): set(np.argsort(-true_pref[u])[:20].tolist()) for u in users}
print(f"{'modelo':<28}{'NDCG observado':>16}{'NDCG vs. gustos reales':>24}")
for name, fn in models.items():
    recs = top_n(fn(users), Rtr[users], 10)
    obs = ranking_metrics(recs, [relevant[u] for u in users], 10)["ndcg"].mean()
    real = ranking_metrics(recs, [relevant_true[u] for u in users], 10)["ndcg"].mean()
    print(f"{name:<28}{obs:>16.3f}{real:>24.3f}")

modelo                        NDCG observado  NDCG vs. gustos reales
popularidad                            0.336                   0.043
FunkSVD (rating predicho)              0.028                   0.418
item-kNN                               0.141                   0.067
PureSVD (k = 20)                       0.113                   0.071
ALS implícito                          0.235                   0.062


**El orden se invierte.** La popularidad, primera en la evaluación offline, es última contra los
gustos reales (0.043); FunkSVD, última offline, es la mejor por mucho (0.418).

Lo que la evaluación offline premia es predecir **qué se va a observar**: aquí, como en la vida real, lo
observado depende de la popularidad (exposición) y no solo del gusto. Los datos de un sistema en
producción traen el sesgo de lo que ese sistema ya mostraba.

## 3. Dos preguntas distintas

FunkSVD responde *“¿cuánto le gustaría si lo viera?”*. Las métricas offline sobre lo observado
responden *“¿qué va a ver y además le va a gustar?”*. Por eso FunkSVD recomienda títulos de nicho
que el usuario valoraría pero que rara vez aparecen en su historial futuro:

In [4]:
m_recs = top_n(models["FunkSVD (rating predicho)"](users), Rtr[users], 10)
p_recs = top_n(models["PureSVD (k = 20)"](users), Rtr[users], 10)
counts = popularity_scores(Rtr)
med_funk, med_svd = np.median(counts[m_recs]), np.median(counts[p_recs])
print(
    f"calificaciones medianas de lo recomendado — FunkSVD: {med_funk:.0f}, PureSVD: {med_svd:.0f}"
)

calificaciones medianas de lo recomendado — FunkSVD: 30, PureSVD: 184


¿Cuál importa? Depende del producto. Si el sistema controla lo que la persona ve (una lista de
reproducción automática), importa el gusto real. Si la persona elige entre muchas opciones y la
recomendación solo compite por su atención, importa también la probabilidad de que la elija. Las
métricas offline mezclan ambas cosas con el sesgo de exposición del sistema anterior; un experimento
A/B es lo único que las separa.

## 4. Datos implícitos: confianza, no preferencia

En la mayoría de sistemas no hay estrellas sino clics, reproducciones o compras. ALS implícito
(Hu, Koren & Volinsky, 2008) trata **toda** celda como observación: 1 si hubo interacción y 0 si no,
pesando cada una con una confianza $c_{ui} = 1 + \alpha r_{ui}$. El “no visto” también enseña algo.

In [5]:
for alpha in [0.1, 1.0, 10.0, 40.0]:
    model = ImplicitALS(k=32, alpha=alpha, n_iter=10).fit(Rtr)
    recs = top_n(model.scores(users), Rtr[users], 10)
    m = ranking_metrics(recs, [relevant[u] for u in users], 10)
    cov = catalog_coverage(recs, shape[1])
    print(f"α = {alpha:>5}: NDCG@10 {m['ndcg'].mean():.3f}, cobertura {cov:.3f}")

α =   0.1: NDCG@10 0.057, cobertura 0.164


α =   1.0: NDCG@10 0.235, cobertura 0.274


α =  10.0: NDCG@10 0.211, cobertura 0.419


α =  40.0: NDCG@10 0.164, cobertura 0.588


## Para llevar

- Evaluar con la tarea real (ranking top-N), no con RMSE, si el producto es una lista.
- La popularidad es un baseline fuerte, en parte por un artefacto: la evaluación offline premia
  predecir qué se observará, no qué gusta. Por eso las decisiones finales se toman con experimentos A/B.
- Reportar cobertura y sesgo de popularidad junto a la precisión: optimizar solo NDCG empuja hacia lo popular.
- Con datos implícitos, ALS ponderado o PureSVD son los puntos de partida clásicos.